In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp
import time

# coins = ["bitcoin", "ethereum", "solana"]
coins = ["bitcoin"]

all_data = []

for coin in coins:
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart?vs_currency=usd&days=1"
    
    for attempt in range(3):
        response = requests.get(url)
        if response.status_code == 200:
            break
        time.sleep(2)
    
    if response.status_code == 200:
        data = response.json()
        
        prices = data["prices"]  # [timestamp, price]
        
        for record in prices:
            all_data.append({
                "coin": coin,
                "timestamp": record[0],
                "price": record[1]
            })
    else:
        print(f"Failed for {coin}")

In [0]:
df = pd.DataFrame(all_data)

In [0]:
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

In [0]:
df.head()

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

spark_df = spark.createDataFrame(df)
spark_df = spark_df.withColumn("ingestion_time", current_timestamp())
spark_df.show(5)

In [0]:
# spark_df.write.format("delta") \
#     .mode("append") \
#     .save("/home/jovyan/CryptoInsight")

In [0]:
# Improve Your Batch Design 
# Add: Partition by date
# 🔥 Why this matters:
# - Faster queries
# - Efficient incremental loads
# - Scalable storage

from pyspark.sql.functions import to_date

spark_df = spark_df.withColumn("date", to_date("timestamp"))

spark_df.show(5)

In [0]:
# spark_df.write.partitionBy("coin") \
#     .format("delta") \
#     .mode("append") \
#     .save("dbfs:/Volumes/workspace/cryptoinsight/bronze")

# spark_df.write.option("header", "true").format("csv").save("bronze")

In [0]:
spark_df.write.partitionBy("coin") \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.cryptoinsight.bronze")

In [0]:
%sql
OPTIMIZE workspace.cryptoinsight.bronze
ZORDER BY (date)